# Validator Judge (LLM 3) — Proof of Concept

A first thin-slice prototype of the **correctness + uncertainty validator** from the design doc.

We **mock the upstream LLMs** (LLM 1 contextualization, LLM 1.2 comprehension, LLM 2 spoiler gate) and feed a fixed
*(reader question, source passage, generated answer)* into **LLM 3, the validator**.

The validator runs the full pipeline on one example:

1. **Decompose** the answer into atomic claims
2. **Route** each claim by grounding source (context / paraphrase / definition / world-knowledge)
3. **Verdict** per claim — Supported / Partially supported / Contradicted / Unverifiable (grounded against the passage)
4. **Aggregate** to an answer-level verdict (worst-case)
5. **Uncertainty** estimate
6. **Map** to a 3-way UI state — Valid / Not reliable / Hedged (Válido / Não confiável / Com ressalvas)

> **Note on uncertainty.** The doc's preferred "cheap single-pass logprob" signal is **not available** — the Anthropic API
> does not expose output-token logprobs. This PoC uses *verbalized confidence* (the doc's named baseline). *N-sample
> consistency* is the principled upgrade and is flagged where it slots in.

# Phase 1

## 0. Dependencies

In [1]:
import subprocess, sys
pkgs = ["anthropic"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)
print("Dependencies OK")

Dependencies OK



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Configuration
Same backend as the anti-spoiler notebook — Anthropic + Haiku for cheap tokens.
Credentials load from Colab `userdata` if present, otherwise from the `ANTHROPIC_API_KEY` env var.

In [2]:
# import os

# # Credentials: Colab userdata -> env var fallback
# API_KEY = None
# try:
#     from google.colab import userdata
#     API_KEY = userdata.get("API_KEY")
# except Exception:
#     API_KEY = os.environ.get("ANTHROPIC_API_KEY")
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not found. Check your .env file.")

# LLM backend
BACKEND         = "anthropic"
ANTHROPIC_MODEL = "claude-sonnet-4-6"  # judge: verdict stage is entailment reasoning -> stronger model helps
OPENAI_MODEL    = "gpt-4o"

print(f"Config  backend={BACKEND}  model={ANTHROPIC_MODEL}  key={'set' if API_KEY else 'MISSING'}")

Config  backend=anthropic  model=claude-sonnet-4-6  key=set


## 2. Model-agnostic LLM client + JSON helper
The same thin `LLMClient` wrapper from the anti-spoiler notebook, plus a small `parse_json_response`
helper that strips markdown fences — every validator stage returns JSON, so we factor that out once.

In [3]:
import json, re

class LLMClient:
    """Thin abstraction over Anthropic / OpenAI.  client.complete(system, user) -> str"""
    def __init__(self, backend: str, api_key: str | None = None):
        self.backend = backend.lower()
        if self.backend == "anthropic":
            import anthropic
            key = api_key or os.environ.get("ANTHROPIC_API_KEY")
            self._client = anthropic.Anthropic(api_key=key)
            self._model  = ANTHROPIC_MODEL
        elif self.backend == "openai":
            import openai
            key = api_key or os.environ.get("OPENAI_API_KEY")
            self._client = openai.OpenAI(api_key=key)
            self._model  = OPENAI_MODEL
        else:
            raise ValueError(f"Unknown backend {backend!r}. Choose anthropic | openai")

    def complete(self, system: str, user: str, max_tokens: int = 1024, temperature: float = 0.0) -> str:
        if self.backend == "anthropic":
            msg = self._client.messages.create(
                model=self._model, max_tokens=max_tokens, temperature=temperature,
                system=system, messages=[{"role": "user", "content": user}],
            )
            return msg.content[0].text.strip()
        else:  # openai
            resp = self._client.chat.completions.create(
                model=self._model, max_tokens=max_tokens, temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
            )
            return resp.choices[0].message.content.strip()


def parse_json_response(raw: str):
    """Strip ```fences``` and parse JSON; surfaces the raw text on failure."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```[\w]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return json.loads(raw)


llm = LLMClient(backend=BACKEND, api_key=API_KEY)
print(f"LLMClient ready  ({BACKEND} / {llm._model})")

LLMClient ready  (anthropic / claude-sonnet-4-6)


## 3. Mock the upstream LLMs — contextualization path (LLM 1 → LLM 2 → LLM 3)

**Interaction model.** The reader never types a free-text question. They (1) **select a passage** and
(2) **click a feature button** — `Define` · `Paraphrase` · `Contextualize` · `Recall` — and the model
infers the request from *(selected text + feature)*. Selection is capped at the reader's position, so
nothing past their current chapter can be selected.

Our example uses **Contextualize** on a selected line about Mr. Bingley — a contextualization request,
so it runs LLM 1 → LLM 2 (spoiler gate) → the validator. We hard-code that path's *output* instead of
running it: the selected text, the feature, the in-bounds grounding passage, and the answer to validate.

The answer is **deliberately seeded** with a mix of claim types — i.e. we plant a known-wrong claim on
purpose (doc §8) so we can confirm the validator catches it:

| # | Claim | Grounding | Expected verdict |
|---|-------|-----------|------------------|
| 1 | Bingley is a single man of large fortune | context (passage) | Supported |
| 2 | He is already engaged to Jane | context (passage) | **Contradicted** (seeded error) |
| 3 | The novel was published in 1813 | world-knowledge | Unverifiable (no web search in PoC) |

Worst-case aggregation should therefore flag the whole answer as **Not reliable**.

In [38]:
# Mocked output of the contextualization path (LLM 1 -> LLM 2 spoiler gate -> validator).
# Interaction model: the reader does NOT type a question. They (1) select a passage and
# (2) click a feature button; the model infers the request from (selected text + feature).
# Feature buttons: "define" | "paraphrase" | "contextualize" | "recall".

READER_POSITION = 15   # reader is through chapter 15 of 61; grounding is limited to <= this

# What the reader actually did: highlighted a span + clicked a feature.
SELECTED_TEXT = "A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
FEATURE       = "contextualize"   # -> contextualization path (characters / places / background)

# Grounding context: stands in for the top-k relevant chunks retrieved from the reader's read-so-far
# scope (chapters <= READER_POSITION). In the final integrated validator this will come from bounded
# retrieval (retrieve_bounded + FAISS, as in the anti-spoiler notebook); we hardcoded here to isolate the judge (D15).
# SOURCE_PASSAGE = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.

# "My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"

# Mr. Bennet replied that he had not. "But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."

# "A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"

# "How so? how can it affect them?"

# "My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
# """
SOURCE_PASSAGE = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"
Mr. Bennet replied that he had not. "But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."
"What is his name?"
"Bingley."
"Is he married or single?"
"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
"How so? how can it affect them?"
"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"""

# Mocked answer the assistant produced for (SELECTED_TEXT, FEATURE), with a seeded error (claim 2).
GENERATED_ANSWER = (
    "Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. "
    "She is especially pleased that he is already engaged to her eldest daughter, Jane. "
    "Pride and Prejudice was first published in 1813."
)

print(f"Reader position : through chapter {READER_POSITION} of 61")
print(f"Feature clicked : {FEATURE}")
print(f"Selected text   : {SELECTED_TEXT}")
print()
print("Generated answer to validate:")
print(" ", GENERATED_ANSWER)

Reader position : through chapter 15 of 61
Feature clicked : contextualize
Selected text   : A single man of large fortune; four or five thousand a year. What a fine thing for our girls!

Generated answer to validate:
  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune. She is especially pleased that he is already engaged to her eldest daughter, Jane. Pride and Prejudice was first published in 1813.


## 4. Decompose & route
Turn the free-text answer into a list of **atomic claims**, each tagged by **grounding source**
(`context` vs `world_knowledge`). No verdicts here — extraction + routing only (decision **D8**).
Rationale for the two-label set and the split-from-verdict choice: see `validator/DECISIONS.md`
(**D7**, **D8**).

**Contract:** `decompose_and_route(answer) -> [{"claim": str, "grounding": "context" | "world_knowledge"}, ...]`

In [39]:
# SYSTEM_DECOMPOSE = """You are the claim-decomposition stage of a validation system for a reading assistant.

# Your job: break an assistant's answer into ATOMIC CLAIMS and tag each by its GROUNDING SOURCE.
# You do NOT judge whether claims are true. Extract faithfully: include every claim, even ones that
# look wrong, and never correct, soften, or rephrase their meaning.

# ATOMIC = one checkable fact per claim. Split conjunctions and compound sentences into separate
# claims. Resolve pronouns and references so each claim stands on its own (e.g. "He" -> "Mr. Bingley").

# GROUNDING SOURCE: tag each claim as exactly one of
# - "context"         : a statement about what happens INSIDE the story (characters, events,
#                       relationships, motivations, setting as described in the book). Must be checked
#                       against the book passage the reader has read.
# - "world_knowledge" : a statement about the real world or the book as a real-world artifact
#                       (historical facts, publication dates, real people/places, literary background).
#                       Would need an external source to verify.

# Output ONLY a JSON array, no prose, no markdown fences. Each element:
#   {"claim": "<the atomic claim as a standalone sentence>", "grounding": "context" | "world_knowledge"}
# """
SYSTEM_DECOMPOSE = """You are the claim-decomposition stage of a validation system for a reading assistant.

Your job: break an assistant's answer into ATOMIC CLAIMS and tag each by its GROUNDING SOURCE.
You do NOT judge whether claims are true. Extract faithfully: include every claim, even ones that
look wrong, and never correct, soften, or rephrase their meaning.

ATOMIC = one checkable fact per claim. Split conjunctions and compound sentences into separate
claims. Resolve pronouns and references so each claim stands on its own (e.g. "He" -> "the lighthouse keeper").

SEPARATE FACTS FROM ATTITUDES/CAUSES. When a sentence ties a feeling, attitude, belief, or reason to
someone ("X is excited because Y", "X is pleased that Y", "X thinks Y"), split it into:
  (a) the underlying factual claim Y, stated plainly, and
  (b) the attitudinal/causal claim linking the person to it.
Each is checked separately — the fact may be in the passage even when the attribution is not.
Example: "The lighthouse keeper was worried because the lamp had gone out" becomes two claims:
  - "The lamp had gone out."
  - "The lighthouse keeper was worried because the lamp had gone out."

GROUNDING SOURCE: tag each claim as exactly one of
- "context"         : a statement about what happens INSIDE the story (characters, events,
                      relationships, motivations, setting as described in the book). Must be checked
                      against the book passage the reader has read.
- "world_knowledge" : a statement about the real world or the book as a real-world artifact
                      (historical facts, publication dates, real people/places, literary background).
                      Would need an external source to verify.

Output ONLY a JSON array, no prose, no markdown fences. Each element:
  {"claim": "<the atomic claim as a standalone sentence>", "grounding": "context" | "world_knowledge"}
"""

def decompose_and_route(answer: str) -> list[dict]:
    """Answer text -> list of {"claim", "grounding"}. Routing only; no verdicts (D8)."""
    raw = llm.complete(SYSTEM_DECOMPOSE, f"ASSISTANT ANSWER TO DECOMPOSE:\n{answer}", max_tokens=600)
    return parse_json_response(raw)

In [9]:
claims = decompose_and_route(GENERATED_ANSWER)
print(f"{len(claims)} atomic claims:")
print()
for i, c in enumerate(claims, 1):
    print(f"{i}. [{c['grounding']:15}] {c['claim']}")

5 atomic claims:

1. [context        ] Mrs. Bennet is excited because Mr. Bingley is a single man.
2. [context        ] Mrs. Bennet is excited because Mr. Bingley has a large fortune.
3. [context        ] Mr. Bingley is already engaged to Jane.
4. [context        ] Jane is Mrs. Bennet's eldest daughter.
5. [world_knowledge] Pride and Prejudice was first published in 1813.


## 5. Grounded per-claim verdict + two uncertainty signals
For each claim: a verdict (`Supported` / `Partially supported` / `Contradicted` / `Unverifiable`, **D9**)
grounded against the right source (**D8**), plus **both** uncertainty signals (**D5**) so we can compare them.

- **Routing:** `world_knowledge` claims short-circuit to `Unverifiable` (no web search in this PoC);
  `context` claims are judged **only** against `SOURCE_PASSAGE`.
- **Anti-memory rule:** the prompt forbids outside knowledge — if the passage doesn't state it, the
  verdict is `Unverifiable` *even if the model believes it's true*. (This is the test claim 4 will face.)
- **Verbalized confidence** (cheap/production): one call at `temperature=0`.
- **N-sample consistency** (offline check): the same prompt re-run `N` times at `temperature>0`;
  uncertainty = modal-verdict **agreement** + label-distribution **entropy**.

Note: 
"Modal" = the most frequent verdict among the `N` samples (the statistical mode).
"Modal-verdict agreement" = the fraction of the `N` samples that landed on that most-common verdict.

Note 2:
Why use modal alongside entropy: they measure the same thing from two angles. Agreement is the intuitive headline number ("how often did it pick the winner"), but it only looks at the top label — it can't tell `C C C U U` apart from `C C C U P` (both 0.60). Entropy looks at the whole distribution, so it captures that the second case is messier. Agreement is easy to read; entropy is the more complete measure. Having both is cheap and lets us see if they ever tell different stories.

In [31]:
import math
from collections import Counter

VERDICT_LABELS = ["Supported", "Partially supported", "Contradicted", "Unverifiable"]

# SYSTEM_VERDICT = """You are the per-claim verdict stage of a validation system for a reading assistant.

# You are given ONE atomic claim and a SOURCE PASSAGE. Decide how the passage bears on the claim.
# Judge ONLY from the passage. Do not use outside knowledge: if the passage does not state or imply the
# claim, the verdict is "Unverifiable" — regardless of whether you personally believe the claim is true
# or false. Entailment against the provided text — not recall from memory — is the whole point.

# Verdict, exactly one of:
# - "Supported"           : the passage states or clearly entails the claim.
# - "Partially supported" : the passage supports part of the claim but not all of it.
# - "Contradicted"        : the passage states or clearly entails the OPPOSITE of the claim.
# - "Unverifiable"        : the passage neither supports nor contradicts the claim (it is silent on it).

# Also report your confidence that your verdict is correct, as a number from 0.0 to 1.0.

# Output ONLY a JSON object, no prose, no markdown fences:
#   {"verdict": "<one label>", "confidence": <0.0-1.0>, "reason": "<one sentence grounded in the passage>"}
# """

SYSTEM_VERDICT = """You are the per-claim verdict stage of a validation system for a reading assistant.

You are given ONE atomic claim and a SOURCE PASSAGE. Decide how the passage bears on the claim.
Judge ONLY from the passage. Do not use outside knowledge: if the passage does not state or imply the
claim, the verdict is "Unverifiable" — regardless of whether you personally believe the claim is true
or false. Entailment against the provided text — not recall from memory — is the whole point.

Verdict, exactly one of:
- "Supported"           : the passage states or clearly entails the claim.
- "Partially supported" : the passage supports part of the claim but not all of it.
- "Contradicted"        : the passage states or clearly entails the OPPOSITE of the claim.
- "Unverifiable"        : the passage neither supports nor contradicts the claim (it is silent on it).

Tie-break (absence vs. contradiction): use "Contradicted" ONLY when the passage explicitly states or
directly entails the opposite of the claim. If the passage simply does not mention the claim, use
"Unverifiable" — even if other statements in the passage loosely suggest the claim might be false.

Also report your confidence that your verdict is correct, as a number from 0.0 to 1.0.

Output ONLY a JSON object, no prose, no markdown fences:
  {"verdict": "<one label>", "confidence": <0.0-1.0>, "reason": "<one sentence grounded in the passage>"}
"""

def judge_claim_once(claim: str, passage: str, temperature: float = 0.0) -> dict:
    """One grounded verdict for a single context claim (judged against the passage)."""
    user = f"SOURCE PASSAGE:\n{passage}\n\nCLAIM:\n{claim}"
    raw = llm.complete(SYSTEM_VERDICT, user, max_tokens=300, temperature=temperature)
    return parse_json_response(raw)

In [14]:
N_SAMPLES   = 5      # N-sample consistency: how many re-samples
SAMPLE_TEMP = 0.8    # temperature for the re-samples (0 would defeat the purpose)

def _entropy(labels: list[str]) -> float:
    """Shannon entropy (bits) of a verdict-label distribution. 0.0 = all samples agree."""
    n = len(labels)
    counts = Counter(labels)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())

def judge_claim(claim: str, grounding: str, passage: str,
                n_samples: int = N_SAMPLES, sample_temp: float = SAMPLE_TEMP) -> dict:
    """Grounded verdict + verbalized confidence + N-sample consistency for one claim."""
    # world_knowledge: no external source in this PoC -> we are *certain* we can't verify it
    if grounding == "world_knowledge":
        return {
            "claim": claim, "grounding": grounding,
            "verdict": "Unverifiable", "verbalized_conf": 1.0,
            "nsample_agreement": None, "nsample_modal": None, "entropy": None,
            "samples": [], "reason": "world-knowledge claim; no external source in this PoC",
        }

    # Verbalized (cheap / production): one pass at temperature 0
    primary = judge_claim_once(claim, passage, temperature=0.0)

    # N-sample (expensive / offline check): N passes at temperature > 0
    samples = [judge_claim_once(claim, passage, temperature=sample_temp)["verdict"]
               for _ in range(n_samples)]
    modal, modal_count = Counter(samples).most_common(1)[0]

    return {
        "claim": claim, "grounding": grounding,
        "verdict": primary["verdict"],                  # production verdict = the single pass
        "verbalized_conf": primary.get("confidence"),
        "nsample_agreement": modal_count / n_samples,   # consistency confidence
        "nsample_modal": modal,
        "entropy": _entropy(samples),
        "samples": samples,
        "reason": primary.get("reason", ""),
    }

In [15]:
SHORT = {"Supported": "S", "Partially supported": "P", "Contradicted": "C", "Unverifiable": "U"}

verdicts = [judge_claim(c["claim"], c["grounding"], SOURCE_PASSAGE) for c in claims]

print(f"Per-claim verdicts ({llm._model}, N={N_SAMPLES} @ temp {SAMPLE_TEMP}):\n")
hdr = f"{'#':>2}  {'verdict':<20} {'verb':>5} {'agree':>6} {'entropy':>8}  samples"
print(hdr); print("-" * len(hdr))
for i, v in enumerate(verdicts, 1):
    verb = f"{v['verbalized_conf']:.2f}" if v["verbalized_conf"] is not None else "  -"
    agr  = f"{v['nsample_agreement']:.2f}" if v["nsample_agreement"] is not None else "   -"
    ent  = f"{v['entropy']:.2f}" if v["entropy"] is not None else "   -"
    samp = " ".join(SHORT.get(s, "?") for s in v["samples"])
    print(f"{i:>2}  {v['verdict']:<20} {verb:>5} {agr:>6} {ent:>8}  {samp}")
    print(f"     [{v['grounding']}] {v['claim']}")

Per-claim verdicts (claude-sonnet-4-6, N=5 @ temp 0.8):

 #  verdict               verb  agree  entropy  samples
-------------------------------------------------------
 1  Partially supported   0.70   1.00    -0.00  P P P P P
     [context] Mrs. Bennet is excited because Mr. Bingley is a single man.
 2  Partially supported   0.70   1.00    -0.00  P P P P P
     [context] Mrs. Bennet is excited because Mr. Bingley has a large fortune.
 3  Unverifiable          0.99   1.00    -0.00  U U U U U
     [context] Mr. Bingley is already engaged to Jane.
 4  Unverifiable          0.97   1.00    -0.00  U U U U U
     [context] Jane is Mrs. Bennet's eldest daughter.
 5  Unverifiable          1.00      -        -  
     [world_knowledge] Pride and Prejudice was first published in 1813.


## 6. Answer-level aggregation (worst-case)
Collapse the per-claim verdicts into one answer-level verdict with the **worst-case** rule (**D10**):
any non-`Supported` claim flags the whole answer. Severity (worst → best):
`Contradicted` > `Unverifiable` > `Partially supported` > `Supported`. The worst claim verdict becomes the
answer verdict, and we surface which claim(s) drove it. (Stage 7 maps this + uncertainty to the 3-way UI state.)

In [17]:
verdicts

[{'claim': 'Mrs. Bennet is excited because Mr. Bingley is a single man.',
  'grounding': 'context',
  'verdict': 'Partially supported',
  'verbalized_conf': 0.7,
  'nsample_agreement': 1.0,
  'nsample_modal': 'Partially supported',
  'entropy': -0.0,
  'samples': ['Partially supported',
   'Partially supported',
   'Partially supported',
   'Partially supported',
   'Partially supported'],
  'reason': "The passage shows Mrs. Bennet is excited about a single man of large fortune potentially marrying one of her daughters, but the name 'Mr. Bingley' is never mentioned in the passage."},
 {'claim': 'Mrs. Bennet is excited because Mr. Bingley has a large fortune.',
  'grounding': 'context',
  'verdict': 'Partially supported',
  'verbalized_conf': 0.7,
  'nsample_agreement': 1.0,
  'nsample_modal': 'Partially supported',
  'entropy': -0.0,
  'samples': ['Partially supported',
   'Partially supported',
   'Partially supported',
   'Partially supported',
   'Partially supported'],
  'reason': 

In [18]:
# Severity for the worst-case rule (D10). Contradicted is worst; Supported best.
# The middle ordering (Unverifiable vs Partially) only affects the displayed label, not the UI bucket.
SEVERITY = {"Supported": 0, "Partially supported": 1, "Unverifiable": 2, "Contradicted": 3}

def aggregate(verdicts: list[dict]) -> dict:
    labels = [v["verdict"] for v in verdicts]
    worst = max(labels, key=lambda l: SEVERITY.get(l, 0))
    flagged = [v for v in verdicts if v["verdict"] == worst and SEVERITY.get(worst, 0) > 0]
    return {"answer_verdict": worst, "counts": Counter(labels),
            "flagged": flagged, "n_claims": len(labels)}

agg = aggregate(verdicts)
print(f"{agg['n_claims']} claims  |  " + ", ".join(f"{k}×{n}" for k, n in agg["counts"].items()))
print(f"\nAnswer-level verdict (worst-case): {agg['answer_verdict']}")
if agg["flagged"]:
    print("Driven by:")
    for v in agg["flagged"]:
        print(f"  - [{v['verdict']}] {v['claim']}")
        print(f"      reason: {v['reason']}")

5 claims  |  Partially supported×2, Unverifiable×3

Answer-level verdict (worst-case): Unverifiable
Driven by:
  - [Unverifiable] Mr. Bingley is already engaged to Jane.
      reason: The passage never mentions Mr. Bingley or Jane by name, nor any engagement between any characters.
  - [Unverifiable] Jane is Mrs. Bennet's eldest daughter.
      reason: The passage makes no mention of Jane or any specific daughters by name; it only refers to 'our girls' without identifying any of them.
  - [Unverifiable] Pride and Prejudice was first published in 1813.
      reason: world-knowledge claim; no external source in this PoC


## 7. Uncertainty gate + 3-way UI mapping
Turn `(answer verdict, uncertainty)` into one of three UI states (**D11**, doc §6):

| answer verdict | uncertainty | UI state |
|---|---|---|
| Supported | low | **Valid** |
| Contradicted | low | **Not reliable** |
| Unverifiable / Partially supported | low | **Hedged** |
| anything | **high** | **Hedged** |

Uncertainty is read from the **verbalized** confidence of the claim(s) that drove the answer verdict
(D5: verbalized is the production signal; N-sample sits beside it in Stage 5). Below `CONF_THRESHOLD`
counts as "high uncertainty" and forces **Hedged** — we don't show a confident Valid/Not-reliable verdict
the judge wasn't sure of.

## 7. Stage 7 — Uncertainty gate + 3-way UI mapping
Turn `(answer verdict, uncertainty)` into one of three UI states (**D11**, doc §6):

| answer verdict | uncertainty | UI state |
|---|---|---|
| Supported | low | **Valid** |
| Contradicted | low | **Not reliable** |
| Unverifiable / Partially supported | low | **Hedged** |
| *(any verdict)* | **high** | **Hedged** |

The last row is an **override**, checked first: if confidence is below threshold, the state is **Hedged**
no matter the verdict — we never show a Valid/Not-reliable the judge was unsure about.

**Which uncertainty signal gates the UI is still open (D5).** We compute *both* answer-level signals —
verbalized confidence and N-sample agreement — and show them side by side. The `UNCERTAINTY_SIGNAL` switch
picks which one drives the gate (default `verbalized`), so you can re-run under each and compare the
end-to-end UI outcome. Neither is hard-wired as "the" production signal — that's what the comparison decides.

In [22]:
UNCERTAINTY_SIGNAL = "verbalized"  # "verbalized" | "nsample" — which signal gates the UI (provisional; D5 open)
VERB_THRESHOLD     = 0.80          # verbalized confidence below this = high uncertainty
AGREE_THRESHOLD    = 0.80          # N-sample agreement below this    = high uncertainty

def _answer_uncertainty(driving: list[dict], key: str):
    """Most-uncertain (min) value of a signal across the driving claims; None if unavailable."""
    vals = [v[key] for v in driving if v.get(key) is not None]
    return min(vals) if vals else None

def map_to_ui(agg: dict, verdicts: list[dict], signal: str = UNCERTAINTY_SIGNAL,
              verb_threshold: float = VERB_THRESHOLD, agree_threshold: float = AGREE_THRESHOLD) -> dict:
    verdict = agg["answer_verdict"]
    driving = agg["flagged"] if agg["flagged"] else verdicts   # claims that set the verdict
    verb  = _answer_uncertainty(driving, "verbalized_conf")
    agree = _answer_uncertainty(driving, "nsample_agreement")

    gate_val, gate_thr = (agree, agree_threshold) if signal == "nsample" else (verb, verb_threshold)
    high_uncertainty = (gate_val is not None) and (gate_val < gate_thr)

    if high_uncertainty:            state = "Hedged"        # override: not confident enough
    elif verdict == "Supported":    state = "Valid"
    elif verdict == "Contradicted": state = "Not reliable"
    else:                           state = "Hedged"        # Unverifiable / Partially supported
    return {"ui_state": state, "answer_verdict": verdict,
            "verbalized_conf": verb, "nsample_agreement": agree,
            "gate_signal": signal, "high_uncertainty": high_uncertainty}

BANNER  = {"Valid": "✅ VALID", "Not reliable": "⛔ NOT RELIABLE", "Hedged": "⚠️ HEDGED"}
MESSAGE = {
    "Valid":        "Show the answer with a 'verified' tag, plus reasoning + sources so the reader can check it.",
    "Not reliable": "\"Couldn't give a reliable response — try again?\" (cap retries).",
    "Hedged":       "\"Couldn't fully verify this.\" Show the answer with reasoning + sources exposed.",
}

ui = map_to_ui(agg, verdicts)
vb = f"{ui['verbalized_conf']:.2f}" if ui["verbalized_conf"] is not None else "n/a"
ag = f"{ui['nsample_agreement']:.2f}" if ui["nsample_agreement"] is not None else "n/a"
print(BANNER[ui["ui_state"]])
print(f"  answer verdict      : {ui['answer_verdict']}")
print(f"  verbalized conf      : {vb}{'   <- gating' if ui['gate_signal']=='verbalized' else ''}")
print(f"  N-sample agreement   : {ag}{'   <- gating' if ui['gate_signal']=='nsample' else ''}")
print(f"  high uncertainty?    : {ui['high_uncertainty']}  (threshold {VERB_THRESHOLD if ui['gate_signal']=='verbalized' else AGREE_THRESHOLD})")
print(f"  UI behaviour         : {MESSAGE[ui['ui_state']]}")

⚠️ HEDGED
  answer verdict      : Unverifiable
  verbalized conf      : 0.97   <- gating
  N-sample agreement   : 1.00
  high uncertainty?    : False  (threshold 0.8)
  UI behaviour         : "Couldn't fully verify this." Show the answer with reasoning + sources exposed.


In [23]:
UNCERTAINTY_SIGNAL = "nsample"  # "verbalized" | "nsample" — which signal gates the UI (provisional; D5 open)
VERB_THRESHOLD     = 0.80          # verbalized confidence below this = high uncertainty
AGREE_THRESHOLD    = 0.80          # N-sample agreement below this    = high uncertainty

def _answer_uncertainty(driving: list[dict], key: str):
    """Most-uncertain (min) value of a signal across the driving claims; None if unavailable."""
    vals = [v[key] for v in driving if v.get(key) is not None]
    return min(vals) if vals else None

ui = map_to_ui(agg, verdicts)
vb = f"{ui['verbalized_conf']:.2f}" if ui["verbalized_conf"] is not None else "n/a"
ag = f"{ui['nsample_agreement']:.2f}" if ui["nsample_agreement"] is not None else "n/a"
print(BANNER[ui["ui_state"]])
print(f"  answer verdict      : {ui['answer_verdict']}")
print(f"  verbalized conf      : {vb}{'   <- gating' if ui['gate_signal']=='verbalized' else ''}")
print(f"  N-sample agreement   : {ag}{'   <- gating' if ui['gate_signal']=='nsample' else ''}")
print(f"  high uncertainty?    : {ui['high_uncertainty']}  (threshold {VERB_THRESHOLD if ui['gate_signal']=='verbalized' else AGREE_THRESHOLD})")
print(f"  UI behaviour         : {MESSAGE[ui['ui_state']]}")

⚠️ HEDGED
  answer verdict      : Unverifiable
  verbalized conf      : 0.97   <- gating
  N-sample agreement   : 1.00
  high uncertainty?    : False  (threshold 0.8)
  UI behaviour         : "Couldn't fully verify this." Show the answer with reasoning + sources exposed.


## 8. Capstone — the validator as one call
`validate(answer, passage)` chains all stages (decompose → route → verdict → aggregate → UI map) and
`report()` prints the result. This is the whole LLM-3 validator behind one function — so a Phase 2
experiment (e.g. P2-2's unambiguous contradiction) is a one-line `validate(new_answer, SOURCE_PASSAGE)`,
not a six-cell re-run.

In [24]:
def validate(answer: str, passage: str, n_samples: int = N_SAMPLES,
             sample_temp: float = SAMPLE_TEMP, signal: str = UNCERTAINTY_SIGNAL) -> dict:
    """Full LLM-3 validator: a generated answer + grounding passage -> verdicts + answer-level UI state."""
    claims   = decompose_and_route(answer)
    verdicts = [judge_claim(c["claim"], c["grounding"], passage, n_samples, sample_temp) for c in claims]
    agg      = aggregate(verdicts)
    ui       = map_to_ui(agg, verdicts, signal=signal)
    return {"claims": claims, "verdicts": verdicts, "aggregate": agg, "ui": ui}

def report(result: dict) -> None:
    ui, agg = result["ui"], result["aggregate"]
    print(f"{agg['n_claims']} claims  |  " + ", ".join(f"{k}×{n}" for k, n in agg["counts"].items()))
    for i, v in enumerate(result["verdicts"], 1):
        vb = f"{v['verbalized_conf']:.2f}" if v["verbalized_conf"] is not None else "  - "
        ag = f"{v['nsample_agreement']:.2f}" if v["nsample_agreement"] is not None else "  - "
        print(f"  {i}. [{v['verdict']:<19}] verb={vb} agree={ag}  {v['claim']}")
    print()
    print(BANNER[ui["ui_state"]], "—", MESSAGE[ui["ui_state"]])
    print(f"  answer verdict {ui['answer_verdict']}  |  gate={ui['gate_signal']}  high_uncertainty={ui['high_uncertainty']}")

# Run the whole validator in one call:
result = validate(GENERATED_ANSWER, SOURCE_PASSAGE)
report(result)

4 claims  |  Partially supported×2, Contradicted×1, Unverifiable×1
  1. [Partially supported] verb=0.70 agree=1.00  Mrs. Bennet is excited because Mr. Bingley is a single man.
  2. [Partially supported] verb=0.70 agree=1.00  Mrs. Bennet is excited because Mr. Bingley has a large fortune.
  3. [Contradicted       ] verb=0.95 agree=1.00  Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to her eldest daughter Jane.
  4. [Unverifiable       ] verb=1.00 agree=  -   Pride and Prejudice was first published in 1813.

⛔ NOT RELIABLE — "Couldn't give a reliable response — try again?" (cap retries).
  answer verdict Contradicted  |  gate=nsample  high_uncertainty=False


# Phase 2

```
Step 1 - baseline:

Ran validate() x10 on the same input:

UI state distribution    : {'Hedged': 4, 'Not reliable': 6}
Claim-count distribution : {5: 9, 4: 1}
Verdict-tuple distribution:
   5x  ('Partially supported', 'Partially supported', 'Contradicted', 'Unverifiable', 'Unverifiable')
   4x  ('Partially supported', 'Partially supported', 'Unverifiable', 'Unverifiable', 'Unverifiable')
   1x  ('Partially supported', 'Partially supported', 'Contradicted', 'Unverifiable')

```

In [ ]:
# Updated SYSTEM_VEREDICT

SYSTEM_VERDICT = """You are the per-claim verdict stage of a validation system for a reading assistant.

You are given ONE atomic claim and a SOURCE PASSAGE. Decide how the passage bears on the claim.
Judge ONLY from the passage. Do not use outside knowledge: if the passage does not state or imply the
claim, the verdict is "Unverifiable" — regardless of whether you personally believe the claim is true
or false. Entailment against the provided text — not recall from memory — is the whole point.

Verdict, exactly one of:
- "Supported"           : the passage states or clearly entails the claim.
- "Partially supported" : the passage supports part of the claim but not all of it.
- "Contradicted"        : the passage states or clearly entails the OPPOSITE of the claim.
- "Unverifiable"        : the passage neither supports nor contradicts the claim (it is silent on it).

Tie-break (absence vs. contradiction): use "Contradicted" ONLY when the passage explicitly states or
directly entails the opposite of the claim. If the passage simply does not mention the claim, use
"Unverifiable" — even if other statements in the passage loosely suggest the claim might be false.

Also report your confidence that your verdict is correct, as a number from 0.0 to 1.0.

Output ONLY a JSON object, no prose, no markdown fences:
  {"verdict": "<one label>", "confidence": <0.0-1.0>, "reason": "<one sentence grounded in the passage>"}
"""

In [30]:
# After changing SYSTEM_VEREDICT
ui_states2, claim_counts2, per_claim_verdicts2 = [], [], []
for _ in range(K):
    r = validate(GENERATED_ANSWER, SOURCE_PASSAGE, n_samples=1)
    ui_states2.append(r["ui"]["ui_state"])
    claim_counts2.append(r["aggregate"]["n_claims"])
    per_claim_verdicts2.append(tuple(v["verdict"] for v in r["verdicts"]))

print(f"Ran validate() x{K} on the same input:\n")
print("UI state distribution    :", dict(_C(ui_states2)))
print("Claim-count distribution :", dict(_C(claim_counts2)))
print("Verdict-tuple distribution:")
for tup, n in _C(per_claim_verdicts2).most_common():
    print(f"  {n:>2}x  {tup}")

Ran validate() x10 on the same input:

UI state distribution    : {'Hedged': 3, 'Not reliable': 7}
Claim-count distribution : {5: 7, 4: 3}
Verdict-tuple distribution:
   4x  ('Partially supported', 'Partially supported', 'Contradicted', 'Unverifiable', 'Unverifiable')
   3x  ('Partially supported', 'Partially supported', 'Unverifiable', 'Unverifiable', 'Unverifiable')
   3x  ('Partially supported', 'Partially supported', 'Contradicted', 'Unverifiable')


In [40]:
# After changing SOURCE_PASSAGE and SYSTEM_DECOMPOSE
result = validate(GENERATED_ANSWER, SOURCE_PASSAGE)
report(result)

7 claims  |  Supported×3, Contradicted×2, Unverifiable×2
  1. [Supported          ] verb=0.99 agree=1.00  Mr. Bingley is a single man.
  2. [Supported          ] verb=0.99 agree=1.00  Mr. Bingley has a large fortune.
  3. [Supported          ] verb=0.97 agree=1.00  Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune.
  4. [Contradicted       ] verb=0.95 agree=1.00  Mr. Bingley is already engaged to Jane.
  5. [Unverifiable       ] verb=0.95 agree=1.00  Jane is Mrs. Bennet's eldest daughter.
  6. [Contradicted       ] verb=0.97 agree=1.00  Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.
  7. [Unverifiable       ] verb=1.00 agree=  -   Pride and Prejudice was first published in 1813.

⛔ NOT RELIABLE — "Couldn't give a reliable response — try again?" (cap retries).
  answer verdict Contradicted  |  gate=nsample  high_uncertainty=False


In [41]:
ui_states3, claim_counts3, per_claim_verdicts3 = [], [], []
for _ in range(K):
    r = validate(GENERATED_ANSWER, SOURCE_PASSAGE, n_samples=1)
    ui_states3.append(r["ui"]["ui_state"])
    claim_counts3.append(r["aggregate"]["n_claims"])
    per_claim_verdicts3.append(tuple(v["verdict"] for v in r["verdicts"]))

print(f"Ran validate() x{K} on the same input:\n")
print("UI state distribution    :", dict(_C(ui_states3)))
print("Claim-count distribution :", dict(_C(claim_counts3)))
print("Verdict-tuple distribution:")
for tup, n in _C(per_claim_verdicts3).most_common():
    print(f"  {n:>2}x  {tup}")

Ran validate() x10 on the same input:

UI state distribution    : {'Not reliable': 10}
Claim-count distribution : {7: 10}
Verdict-tuple distribution:
  10x  ('Supported', 'Supported', 'Supported', 'Contradicted', 'Unverifiable', 'Contradicted', 'Unverifiable')


In [42]:
# Diagnostic: does temperature actually vary the verdict on claude-sonnet-4-6?
# Same claim + passage, K calls at each temperature.
diag_claim = "Mr. Bingley is already engaged to Jane."   # the borderline claim
for temp in (0.0, 0.8, 1.0):
    outs = [judge_claim_once(diag_claim, SOURCE_PASSAGE, temperature=temp)["verdict"] for _ in range(10)]
    print(f"temp={temp:<3}: {dict(Counter(outs))}")

temp=0.0: {'Contradicted': 10}
temp=0.8: {'Contradicted': 10}
temp=1.0: {'Contradicted': 10}


In [43]:
# Is temperature functional in our client at all?
for temp in (0.0, 1.0):
    outs = [llm.complete("Reply with exactly one common animal name and nothing else.",
                         "Name an animal.", max_tokens=10, temperature=temp)
            for _ in range(8)]
    print(f"temp={temp}: {outs}")

temp=0.0: ['Cat', 'Cat', 'Cat', 'Dog', 'Dog', 'Dog', 'Dog', 'Cat']
temp=1.0: ['Dog', 'Cat', 'Cat', 'Dog', 'Cat', 'Cat', 'Cat', 'Cat']


# Phase 3

In [44]:
# Making K cross-run parallel 

from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items, max_workers=6):
    """Run fn over items concurrently (I/O-bound LLM calls); preserves order."""
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(fn, items))

In [45]:
K = 10

def validate(answer: str, passage: str, n_samples: int = N_SAMPLES,
             sample_temp: float = SAMPLE_TEMP, signal: str = UNCERTAINTY_SIGNAL) -> dict:
    """Full LLM-3 validator: a generated answer + grounding passage -> verdicts + answer-level UI state."""
    claims   = decompose_and_route(answer)
    verdicts = parallel_map(lambda c: judge_claim(c["claim"], c["grounding"], passage), claims)
    agg      = aggregate(verdicts)
    ui       = map_to_ui(agg, verdicts, signal=signal)
    return {"claims": claims, "verdicts": verdicts, "aggregate": agg, "ui": ui}

def report(result: dict) -> None:
    ui, agg = result["ui"], result["aggregate"]
    print(f"{agg['n_claims']} claims  |  " + ", ".join(f"{k}×{n}" for k, n in agg["counts"].items()))
    for i, v in enumerate(result["verdicts"], 1):
        vb = f"{v['verbalized_conf']:.2f}" if v["verbalized_conf"] is not None else "  - "
        ag = f"{v['nsample_agreement']:.2f}" if v["nsample_agreement"] is not None else "  - "
        print(f"  {i}. [{v['verdict']:<19}] verb={vb} agree={ag}  {v['claim']}")
    print()
    print(BANNER[ui["ui_state"]], "—", MESSAGE[ui["ui_state"]])
    print(f"  answer verdict {ui['answer_verdict']}  |  gate={ui['gate_signal']}  high_uncertainty={ui['high_uncertainty']}")

# Run the whole validator in one call:
# result = validate(GENERATED_ANSWER, SOURCE_PASSAGE)
results = parallel_map(lambda _: validate(GENERATED_ANSWER, SOURCE_PASSAGE, n_samples=1), range(K), max_workers=K)

In [46]:
results

[{'claims': [{'claim': 'Mr. Bingley is a single man.', 'grounding': 'context'},
   {'claim': 'Mr. Bingley has a large fortune.', 'grounding': 'context'},
   {'claim': 'Mrs. Bennet is excited because Mr. Bingley is a single man with a large fortune.',
    'grounding': 'context'},
   {'claim': 'Mr. Bingley is already engaged to Jane.',
    'grounding': 'context'},
   {'claim': "Jane is Mrs. Bennet's eldest daughter.", 'grounding': 'context'},
   {'claim': 'Mrs. Bennet is especially pleased that Mr. Bingley is already engaged to Jane.',
    'grounding': 'context'},
   {'claim': 'Pride and Prejudice was first published in 1813.',
    'grounding': 'world_knowledge'}],
  'verdicts': [{'claim': 'Mr. Bingley is a single man.',
    'grounding': 'context',
    'verdict': 'Supported',
    'verbalized_conf': 0.99,
    'nsample_agreement': 1.0,
    'nsample_modal': 'Supported',
    'entropy': -0.0,
    'samples': ['Supported',
     'Supported',
     'Supported',
     'Supported',
     'Supported'],

In [48]:
ui_states4, claim_counts4, per_claim_verdicts4 = [], [], []
for r in results:
    ui_states4.append(r["ui"]["ui_state"])
    claim_counts4.append(r["aggregate"]["n_claims"])
    per_claim_verdicts4.append(tuple(v["verdict"] for v in r["verdicts"]))

print(f"Ran validate() x{K} on the same input:\n")
print("UI state distribution    :", dict(_C(ui_states4)))
print("Claim-count distribution :", dict(_C(claim_counts4)))
print("Verdict-tuple distribution:")
for tup, n in _C(per_claim_verdicts4).most_common():
    print(f"  {n:>2}x  {tup}")

Ran validate() x10 on the same input:

UI state distribution    : {'Not reliable': 10}
Claim-count distribution : {7: 10}
Verdict-tuple distribution:
  10x  ('Supported', 'Supported', 'Supported', 'Contradicted', 'Unverifiable', 'Contradicted', 'Unverifiable')
